# Math Reasoning LLM - Operational Launcher

This notebook serves as the operational launcher for training, evaluation, and data generation.
It works seamlessly both **in Google Colab** (with T4/G4/A100 GPUs) and **in Antigravity IDE / local Jupyter**.

> **Architecture Rules:**
> 1. **No Training Logic in Notebook Cells:** All post-training, data synthesis, and evaluation runs inside modular scripts in `scripts/`.
> 2. **Persistence on Google Drive (Colab mode):** Artifacts (`checkpoints/`, `outputs/`, `logs/`, `data/`) are saved directly to `My Drive/math-reasoning-llm/`.
> 3. **High-Speed Model Caching:** Hugging Face model cache remains on fast local SSD disk (`/content/hf_cache` on Colab, or local `.cache/`), NEVER on Google Drive.
> 4. **Zero-Secret Commits:** Hugging Face token is loaded directly from environment or Secrets (`HF_TOKEN`).

## Step 1: Environment Detection & Persistence Setup
Detects whether running in Google Colab (mounts Google Drive) or local workstation, and sets the root directory.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure current working directory is the repository root, not notebooks/
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
print(f"Active working directory: {Path.cwd()}")

# Check if running inside Google Colab
IS_COLAB = "google.colab" in sys.modules
try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    print("Running in Google Colab. Mounting Google Drive...")
    drive.mount('/content/drive')
    DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/math-reasoning-llm')
    for subfolder in ['checkpoints', 'outputs', 'logs', 'data']:
        (DRIVE_PROJECT_ROOT / subfolder).mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted successfully at: {DRIVE_PROJECT_ROOT}")
else:
    print("Running in local/IDE environment. Using local workspace storage.")
    for subfolder in ['checkpoints', 'outputs', 'logs', 'data']:
        Path(subfolder).mkdir(parents=True, exist_ok=True)
    print("Local directories verified.")

## Step 2: Repository Branch Synchronization
Ensures latest project code is checked out.

In [ ]:
REPO_URL = "https://github.com/NashLucas/math-reasoning-llm.git"
REPO_DIR = "/content/math-reasoning-llm"
BRANCH = "phase-2-generation-and-baseline"

if IS_COLAB:
    if not os.path.exists(REPO_DIR):
        print(f"Cloning {REPO_URL}...")
        !git clone {REPO_URL} {REPO_DIR}
        %cd {REPO_DIR}
        !git checkout {BRANCH}
    else:
        print(f"Updating repository in {REPO_DIR}...")
        %cd {REPO_DIR}
        !git fetch origin
        !git checkout {BRANCH}
        !git pull origin {BRANCH}
    %cd {REPO_DIR}
else:
    print("Local workspace already loaded.")

!git status

## Step 3: Hardware Diagnostics & Environment Configuration
Inspects GPU hardware (CUDA, VRAM) and configures HF cache.

In [ ]:
try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

# 1. HF Token
if IS_COLAB:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        os.environ['HF_TOKEN'] = hf_token
        os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
        print("HF token loaded from Colab Secrets.")
    except Exception:
        print("NOTE: HF_TOKEN not set in Colab Secrets (optional for public models like Qwen / DeepSeek-Math).")
    os.environ['HF_HOME'] = '/content/hf_cache'
    os.environ['APP_PROFILE'] = 'colab'
    os.makedirs('/content/hf_cache', exist_ok=True)

# 2. Diagnostics
print("=" * 60)
print("HARDWARE DIAGNOSTICS")
print("=" * 60)
if HAS_TORCH and torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Detected: {gpu_name}")
    print(f"VRAM: {vram_gb:.2f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"PyTorch Version: {torch.__version__}")
else:
    print("CPU Runtime detected (No CUDA GPU active).")
    if IS_COLAB:
        print("To enable GPU on Colab: Runtime -> Change runtime type -> T4 GPU.")
print("=" * 60)

## Step 4: Install Dependencies

In [ ]:
if IS_COLAB:
    !pip install -r requirements.txt
else:
    print("Dependencies managed by local environment.")

## Step 5: Verify Configurations & Stage Paths

In [ ]:
!python src/config.py

## Step 6: Day 11 Verification — Development Model Generation & Chunk Resumption

- On GPU (Colab T4): runs with `--use_real_model` to load `Qwen/Qwen2.5-1.5B-Instruct` on GPU.
- On CPU (Local): runs without `--use_real_model` to test stop rules and resumption with the mock generator.

In [ ]:
# If on GPU runtime, use real model; otherwise use mock generator
if HAS_TORCH and torch.cuda.is_available():
    print("Running Day 11 verification with real Qwen2.5-1.5B-Instruct on GPU...")
    !python scripts/test_generation_engine.py --use_real_model --model_name "Qwen/Qwen2.5-1.5B-Instruct"
else:
    print("Running Day 11 verification in mock mode (CPU / local test)...")
    !python scripts/test_generation_engine.py

## Step 7: Day 12 Verification — Evaluation Engine & Metrics on Sample Data

Tests `scripts/evaluate.py` end-to-end:
- Runs greedy evaluation on the 20 sample problems (`val_sample.parquet`)
- Computes math metrics (accuracy, format compliance, fallback rate, token averages)
- Generates `eval_results.json` and `run_info.json` in `outputs/baseline/`

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 12 evaluation on sample data with real Qwen2.5-1.5B on GPU...')
    !python scripts/evaluate.py --stage baseline --preset greedy --split sample --limit 20
else:
    print('Running Day 12 evaluation on sample data in mock mode (CPU / local test)...')
    !python scripts/evaluate.py --stage baseline --preset greedy --split sample --limit 20 --use_mock


## Step 8: Day 13 — Development Model Baseline (500 Validation Problems)

Runs greedy and self-consistency decoding across all 500 validation problems (`val.parquet`):
- Generates `outputs/baseline/eval_results_val_greedy.json` & `.jsonl`
- Generates `outputs/baseline/eval_results_val_self_consistency.json` & `.jsonl`
- Generates `outputs/baseline/run_info.json`
- High-throughput inference via vLLM (or Hugging Face batched generation)

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 13 development baseline on 500 validation problems (GPU)...')
    !python scripts/run_dev_baseline.py --split val
else:
    print('Running Day 13 development baseline in mock mode (CPU / local test)...')
    !python scripts/run_dev_baseline.py --split val --limit 20 --use_mock


## Step 9: Day 13 — Output Audit & Parser Quality Review

Inspects 20 model reasoning completions to verify answer extraction, format compliance, and equivalence checking.
Generates `reports/day13_output_review.md`.

In [ ]:
!python scripts/review_outputs.py --count 20


## Step 10: Day 14 — 7B Baselines (3 Models x 2 Presets on 500 Validation Problems)

Executes baseline evaluations for all three DeepSeek-Math 7B models on `val.parquet` (500 problems):
1. `deepseek-ai/deepseek-math-7b-base` (8-shot few-shot prompt)
2. `deepseek-ai/deepseek-math-7b-instruct` (chat template)
3. `deepseek-ai/deepseek-math-7b-rl` (chat template)

Under `greedy` and `self_consistency` decoding presets.
Outputs 6 result sets in `outputs/baseline/<model_slug>/` and summary table `outputs/baseline/baseline_comparison.md`.

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 14 7B baselines on GPU (G4/T4)...')
    !python scripts/run_7b_baselines.py --split val
else:
    print('Running Day 14 7B baselines in mock mode (CPU / local test)...')
    !python scripts/run_7b_baselines.py --split val --limit 20 --use_mock


## Step 11: Day 15 — Safety Prompt Set & Llama Guard Screening

1. Builds `data/safety/` (50 JailbreakBench harmful prompts, 5 per category, + 50 benign GSM8K train questions, seed 42).
2. Runs the safety unit tests.
3. On a GPU with at least 20 GB VRAM (L4 / G4 / A100), generates development-model responses and screens them with `meta-llama/Llama-Guard-3-8B`.
   Llama Guard 3 is gated: `HF_TOKEN` must belong to an account with approved access. A T4 (15 GB) cannot hold it in bf16.

Outputs `outputs/baseline/safety/<model_slug>/safety_results.json`, `safety_assessments.jsonl` and `run_info.json`.

In [ ]:
!python scripts/prepare_safety_data.py
!python -m unittest tests/test_safety.py

if HAS_TORCH and torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    if vram_gb >= 20:
        print(f'Running Day 15 safety evaluation on the development model ({vram_gb:.0f} GB GPU)...')
        !python scripts/run_safety_eval.py
    else:
        print(f'GPU has {vram_gb:.0f} GB; Llama Guard 3 8B needs ~16 GB plus headroom. Switch to L4/G4/A100.')
else:
    print('Running Day 15 safety evaluation in mock mode (CPU / local test)...')
    !python scripts/run_safety_eval.py --use_mock --limit 10 --output_dir outputs/mock/safety


## Step 12: Day 15 — Manual Review of 30 Outputs per 7B Model

Writes one review report per model to `reports/day15_review_<model_slug>.md` from the **real** Step 10 greedy generations.
Read every row and check that the parsed answer matches what you would read; add every parser miss to `tests/test_answer_extraction.py` before fixing it.

In [ ]:
for slug in ['deepseek-math-7b-base', 'deepseek-math-7b-instruct', 'deepseek-math-7b-rl']:
    gen_file = f'outputs/baseline/{slug}/generations_val_greedy.jsonl'
    !python scripts/review_outputs.py --input_file {gen_file} --count 30 --output_report reports/day15_review_{slug}.md
